# MNIST Classification with blox and Distrax

This tutorial demonstrates how to train a Convolutional Neural Network (CNN) on MNIST using **blox**.

We will strictly follow a probabilistic approach:
1.  **Model**: Defines a conditional distribution $P(Y | X)$.
2.  **Objective**: Maximize the likelihood of the data (minimize Negative Log Likelihood).

We use **Distrax** to handle the probability distributions.

In [ ]:
# Install necessary packages.
!pip install -q jax-blox optax tensorflow-datasets matplotlib distrax

In [ ]:
import jax
import jax.numpy as jnp
import blox as bx
import optax
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np
import distrax

In [ ]:
def get_datasets():
  """Load MNIST train and test datasets into memory."""
  ds_builder = tfds.builder('mnist')
  ds_builder.download_and_prepare()
  train_ds = tfds.as_numpy(ds_builder.as_dataset(split='train', batch_size=-1))
  test_ds = tfds.as_numpy(ds_builder.as_dataset(split='test', batch_size=-1))

  # Normalize to [0, 1] and add channel dimension if missing (N, H, W, C).
  train_ds['image'] = jnp.float32(train_ds['image']) / 255.0
  test_ds['image'] = jnp.float32(test_ds['image']) / 255.0
  return train_ds, test_ds


train_ds, test_ds = get_datasets()
print(
  f'Train shape: {train_ds["image"].shape}, Label shape: {train_ds["label"].shape}'
)

In [ ]:
class CNN(bx.Module):
  """A probabilistic CNN classifier."""

  def __init__(self, graph: bx.Graph, num_classes: int = 10):
    super().__init__(graph)
    self.conv1 = bx.Conv(
      graph.child('conv1'), output_channels=32, kernel_size=3
    )
    self.conv2 = bx.Conv(
      graph.child('conv2'), output_channels=64, kernel_size=3
    )
    self.linear1 = bx.Linear(graph.child('linear1'), output_size=256)
    self.linear2 = bx.Linear(graph.child('linear2'), output_size=num_classes)
    self.dropout = bx.Dropout(graph.child('dropout'), rate=0.5)

  def __call__(self, params: bx.Params, x: jax.Array, is_training: bool = True):
    # Convolutional feature extraction.
    x, params = self.conv1(params, x)
    x = jax.nn.relu(x)
    x = bx.avg_pool(x, window_shape=2, strides=2)

    x, params = self.conv2(params, x)
    x = jax.nn.relu(x)
    x = bx.avg_pool(x, window_shape=2, strides=2)

    # Flatten and dense layers.
    x = x.reshape((x.shape[0], -1))
    x, params = self.linear1(params, x)
    x = jax.nn.relu(x)
    x, params = self.dropout(params, x, is_training=is_training)

    # Output logits for the Categorical distribution.
    logits, params = self.linear2(params, x)
    return logits, params

In [ ]:
def initialize_model(seed=0):
  graph = bx.Graph('mnist_cnn')
  model = CNN(graph)
  rng = bx.Rng(graph.child('rng'), seed=seed)
  params = bx.Params(rng=rng)

  # Lazy initialization pass.
  dummy_input = jnp.ones((1, 28, 28, 1))
  _, params = model(params, dummy_input, is_training=False)
  return model, params.finalize()


model, params = initialize_model()
bx.display(model.graph, params)

In [ ]:
@jax.jit
def train_step(params, opt_state, batch_images, batch_labels, optimizer):
  trainable, non_trainable = params.split()

  def loss_fn(t_params):
    curr_params = t_params.merge(non_trainable)
    logits, new_params = model(curr_params, batch_images, is_training=True)

    # Probabilistic Loss: Negative Log Likelihood.
    # We model the output as a Categorical distribution.
    dist = distrax.Categorical(logits=logits)
    nll = -dist.log_prob(batch_labels).mean()

    _, new_non_trainable = new_params.split()
    return nll, new_non_trainable

  (loss, new_non_trainable), grads = jax.grad(loss_fn, has_aux=True)(trainable)
  updates, new_opt_state = optimizer.update(grads, opt_state, trainable)
  new_trainable = optax.apply_updates(trainable, updates)

  return new_trainable.merge(new_non_trainable), new_opt_state, loss

In [ ]:
@jax.jit
def eval_step(params, batch_images, batch_labels):
  logits, _ = model(params, batch_images, is_training=False)
  # The mode of the distribution is the prediction.
  dist = distrax.Categorical(logits=logits)
  predicted_class = dist.mode()
  accuracy = jnp.mean(predicted_class == batch_labels)
  return accuracy

In [ ]:
def train_model(num_epochs=5, batch_size=32, learning_rate=1e-3):
  model, params = initialize_model()
  optimizer = optax.adamw(learning_rate)
  trainable_params, _ = params.split()
  opt_state = optimizer.init(trainable_params)

  history = {'loss': [], 'val_acc': []}

  for epoch in range(num_epochs):
    num_train = train_ds['image'].shape[0]
    perm = np.random.permutation(num_train)
    X, Y = train_ds['image'][perm], train_ds['label'][perm]

    epoch_losses = []
    for i in range(0, num_train, batch_size):
      batch_X = X[i : i + batch_size]
      batch_Y = Y[i : i + batch_size]
      if batch_X.shape[0] < batch_size:
        continue

      params, opt_state, loss = train_step(
        params, opt_state, batch_X, batch_Y, optimizer
      )
      epoch_losses.append(loss)

    avg_loss = np.mean(epoch_losses)
    history['loss'].append(avg_loss)

    # Evaluation.
    test_accs = []
    for i in range(0, test_ds['image'].shape[0], batch_size):
      batch_X = test_ds['image'][i : i + batch_size]
      batch_Y = test_ds['label'][i : i + batch_size]
      test_accs.append(eval_step(params, batch_X, batch_Y))
    test_acc = np.mean(test_accs)
    history['val_acc'].append(test_acc)

    print(f'Epoch {epoch + 1}, Loss: {avg_loss:.4f}, Test Acc: {test_acc:.4f}')

  return params, history


trained_params, history = train_model()

In [ ]:
# Visualize Training Progress.
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history['loss'], label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Negative Log Likelihood')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['val_acc'], label='Test Accuracy', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()


# Visualize Predictions.
def show_predictions(params, count=5):
  images = test_ds['image'][:count]
  labels = test_ds['label'][:count]
  logits, _ = model(params, images, is_training=False)

  # Probabilistic prediction.
  dist = distrax.Categorical(logits=logits)
  preds = dist.mode()

  fig, axes = plt.subplots(1, count, figsize=(15, 3))
  for i, ax in enumerate(axes):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(f'True: {labels[i]}, Pred: {preds[i]}')
    ax.axis('off')
  plt.show()


show_predictions(trained_params)